# Phase 3 — Sequential Frame Processing

**Problem:** VGGT uses global self-attention with O(N²) complexity.
Processing 50 frames needs ~11 GB, 100 frames ~21 GB, 200 frames ~40 GB —
all exceeding free-tier GPU limits.

**Solution (sliding window):**
1. Split sequence into overlapping chunks
2. Run VGGT per chunk (each fits in 16 GB)
3. Align consecutive chunks via Procrustes on overlap camera centres
4. Merge into a single world-frame reconstruction

This notebook covers:
1. Problem demonstration (memory vs. frame count)
2. Sliding window algorithm walkthrough
3. Chunk size experiments (5, 10, 15, 20 frames)
4. Overlap experiments (1, 2, 3 frames)
5. Memory profiling & trade-off analysis
6. Quality comparison: chunked vs. full-batch
7. Visualisation of chunk-coloured point clouds

## 1. Setup

In [ ]:
import sys, os

IN_COLAB  = 'google.colab' in sys.modules
IN_KAGGLE = os.path.exists('/kaggle')

if IN_COLAB or IN_KAGGLE:
    import subprocess
    for p in [
        'numpy==1.26.4',
        'torch torchvision --index-url https://download.pytorch.org/whl/cu118',
        'plotly matplotlib open3d trimesh huggingface_hub',
        'git+https://github.com/jytime/LightGlue.git',
    ]:
        subprocess.run(f'pip install -q {p}', shell=True)
    if not os.path.exists('vggt'):
        subprocess.run('git clone --depth 1 https://github.com/facebookresearch/vggt.git', shell=True)
    subprocess.run('pip install -q -r vggt/requirements.txt', shell=True)
    # Make the vggt package importable in this kernel without needing a restart
    vggt_root = os.path.abspath('vggt')
    if vggt_root not in sys.path:
        sys.path.insert(0, vggt_root)
    if not os.path.exists('vggt-eval'):
        subprocess.run('git clone --depth 1 https://github.com/insha-py/vggt-eval.git', shell=True)
    eval_root = os.path.abspath('vggt-eval')
    if eval_root not in sys.path:
        sys.path.insert(0, eval_root)

# Quick import smoke-test
try:
    from vggt.models.vggt import VGGT
    print('vggt package found OK')
except ModuleNotFoundError as e:
    print(f'WARNING: {e} — check that the vggt clone succeeded above.')

import torch
import numpy as np
import matplotlib.pyplot as plt
import glob

REPO_ROOT = os.path.abspath(os.path.join(os.path.dirname(''), '..'))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from src.pipeline      import VGGTPipeline, load_images_from_list, _list_images
from src.chunking      import (
    SlidingWindowProcessor, experiment_chunk_sizes, experiment_overlaps,
    profile_memory_theoretical, align_chunk_to_reference,
)
from src.visualization import (
    plot_point_cloud, plot_chunk_alignment, plot_memory_vs_frames,
    plot_cameras, save_ply,
)
from src.metrics import MemoryProfiler, Timer, print_metrics_table

# ── Configuration ──────────────────────────────────────────────
IMAGE_DIR   = None           # set to real image folder; None → synthetic
OUTPUT_DIR  = '../results'
CHECKPOINT  = None           # local path or None to download
CONF_THRESH = 5.0
MAX_FRAMES  = 60             # total frames to use for experiments
CHUNK_SIZE  = 10             # default chunk size
OVERLAP     = 2              # default overlap

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Setup complete.')

In [ ]:
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU : {props.name}  |  VRAM: {props.total_memory/1024**3:.1f} GB')
else:
    print('WARNING: No GPU detected.')

## 2. The Problem — Memory Scales with O(N²)

Before running experiments, let's visualise the memory scaling issue
using the empirical numbers from the VGGT paper (Table 9).

In [ ]:
from src.chunking import profile_memory_theoretical

frame_counts = [1, 5, 10, 20, 30, 50, 75, 100, 150, 200]
mem_estimates = profile_memory_theoretical(frame_counts)

ns   = [r['n_frames']     for r in mem_estimates]
gbs  = [r['estimated_gb'] for r in mem_estimates]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(ns, gbs, 'b-o', lw=2, ms=6, label='Estimated GPU memory')
ax.axhline(y=16, color='red',    ls='--', lw=2, label='16 GB limit (T4/P100)')
ax.axhline(y=40, color='orange', ls='--', lw=2, label='40 GB (A100)')

# Annotate published values from Table 9
for n, gb in [(1, 1.88/1), (10, 3.63), (50, 11.41), (200, 40.63)]:
    ax.annotate(f'{gb:.1f} GB', xy=(n, gb), xytext=(n+5, gb+1),
                fontsize=8, color='darkblue',
                arrowprops=dict(arrowstyle='->', color='darkblue'))

ax.set_xlabel('Number of input frames')
ax.set_ylabel('GPU memory (GB)')
ax.set_title('VGGT Memory Scaling (O(N²) attention)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'memory_scaling.png'), dpi=150)
plt.show()

print('\n  N frames   Est. GB')
print('  ─────────────────')
for r in mem_estimates:
    over = ' ← OOM on T4' if r['estimated_gb'] > 16 else ''
    print(f'  {r["n_frames"]:>8d}   {r["estimated_gb"]:>6.1f}{over}')

## 3. Prepare Image Sequence

We need a longer sequence to demonstrate the value of chunking.
If no real dataset is available, we generate synthetic frames.

In [ ]:
from PIL import Image as PILImage

def make_synthetic_sequence(out_dir, n=60, size=256):
    """Generate a synthetic forward-motion image sequence."""
    os.makedirs(out_dir, exist_ok=True)
    cx, cy = size // 2, size // 2
    for i in range(n):
        arr = np.zeros((size, size, 3), dtype=np.uint8)
        # Simulate forward motion: shrinking rectangle + background gradient
        progress = i / max(n - 1, 1)
        r = int(size * 0.4 * (1 - 0.5 * progress))
        arr[:, :, 0] = np.uint8(progress * 200)
        arr[:, :, 2] = np.uint8((1 - progress) * 200)
        if r > 0:
            arr[cy-r:cy+r, cx-r:cx+r, 1] = 200
        PILImage.fromarray(arr).save(os.path.join(out_dir, f'frame_{i:04d}.png'))
    return out_dir

if IMAGE_DIR is None:
    print('Generating synthetic sequence for demonstration.')
    IMAGE_DIR = make_synthetic_sequence(
        '../datasets/synthetic/long_sequence', n=MAX_FRAMES
    )

all_paths = _list_images(IMAGE_DIR)[:MAX_FRAMES]
print(f'Using {len(all_paths)} frames from {IMAGE_DIR}')

## 4. Load Model

In [ ]:
pipe = VGGTPipeline()
pipe.load_model(checkpoint_path=CHECKPOINT)

## 5. Baseline: Full-Batch Inference

Process all frames at once (only possible if it fits in VRAM).
This gives us a reference quality for comparison.

In [ ]:
# Try full-batch; gracefully handle OOM
baseline_result = None

# Only attempt if estimated memory is within 90% of available VRAM
estimated_gb = profile_memory_theoretical([len(all_paths)])[0]['estimated_gb']
vram_gb = (torch.cuda.get_device_properties(0).total_memory / 1024**3
           if torch.cuda.is_available() else 0)

print(f'Estimated memory for {len(all_paths)} frames: {estimated_gb:.1f} GB')
print(f'Available VRAM: {vram_gb:.1f} GB')

if estimated_gb < 0.9 * vram_gb:
    print('Running full-batch inference as baseline...')
    try:
        baseline_result = pipe.run(IMAGE_DIR, max_frames=len(all_paths),
                                   conf_thresh=CONF_THRESH)
        print(f'  Baseline: {len(baseline_result["point_cloud"]):,} points  '
              f'{baseline_result["inference_time_s"]:.2f}s  '
              f'{baseline_result["peak_gpu_mb"]:.0f} MB')
    except torch.cuda.OutOfMemoryError:
        print('OOM: full-batch is too large — this is exactly why we need chunking!')
        torch.cuda.empty_cache()
else:
    print('Skipping full-batch (would exceed VRAM) — chunked result will be the main output.')

## 6. Sliding-Window Processing

### How the algorithm works

```
Frames: [0  1  2  3  4  5  6  7  8  9  10 11 ...]

Chunk 0:  [0  1  2  3  4  5  6  7  8  9]          ← defines world frame
Chunk 1:          [6  7  8  9  10 11 12 13 14 15]  ← overlap = frames 6,7,8,9
Chunk 2:                 [12 13 14 15 16 17 18 19]  ← overlap = frames 12,13,14,15
```

**Alignment:** Procrustes on overlap camera centres → rigid transform (R, t, s)
applied to all extrinsics and 3-D points of the new chunk.

In [ ]:
proc = SlidingWindowProcessor(
    pipe,
    chunk_size=CHUNK_SIZE,
    overlap=OVERLAP,
    conf_thresh=CONF_THRESH,
)

chunked_result = proc.process(image_paths=all_paths, verbose=True)

print(f'\nMerged result:')
print(f'  Total frames  : {len(chunked_result["extrinsic"])}')
print(f'  Total points  : {len(chunked_result["point_cloud"]):,}')
print(f'  Chunks        : {len(chunked_result["chunk_results"])}')

## 7. Visualise Chunk-Coloured Reconstruction

Each chunk is rendered in a distinct colour so we can inspect alignment quality.

In [ ]:
from src.chunking import transform_point_cloud

# Build per-chunk aligned point lists
chunks_pts  = []
chunks_extrs = []

# Chunk 0 is already in world frame
cr0 = chunked_result['chunk_results'][0]
chunks_pts.append(cr0.point_cloud)
chunks_extrs.append(cr0.extrinsic)

# Remaining chunks: use the alignment info stored in chunk_stats
for ci, (cr, info) in enumerate(
    zip(chunked_result['chunk_results'][1:], chunked_result['chunk_stats'][1:]),
    start=1,
):
    R_T, t_T, s_T = info['R'], info['t'], info['s']
    pts_aligned = transform_point_cloud(cr.point_cloud, R_T, t_T, s_T)
    chunks_pts.append(pts_aligned)
    # Aligned extrinsics for this chunk are in chunked_result['extrinsic']
    start_i = (CHUNK_SIZE - OVERLAP) * ci
    end_i   = start_i + (cr.frame_end - cr.frame_start)
    chunks_extrs.append(chunked_result['extrinsic'][start_i:end_i])

fig_chunks = plot_chunk_alignment(
    chunks_pts,
    chunks_extrs,
    title=f'Chunk-wise Reconstruction (chunk_size={CHUNK_SIZE}, overlap={OVERLAP})',
    save_html=os.path.join(OUTPUT_DIR, 'figures', 'chunk_alignment.html'),
    show=True,
)

In [ ]:
# Merged point cloud
fig_merged = plot_point_cloud(
    chunked_result['point_cloud'],
    colors=chunked_result['point_colors'],
    title=f'Merged Reconstruction ({len(chunked_result["point_cloud"]):,} points)',
    save_html=os.path.join(OUTPUT_DIR, 'figures', 'merged_point_cloud.html'),
    show=True,
)

## 8. Alignment Quality per Chunk

In [ ]:
stats = chunked_result['chunk_stats']

print(f'\n{"─"*60}')
print(f'  {"Chunk":>6}  {"Residual (m)":>14}  {"Scale":>8}  {"OV frames":>10}')
print(f'{"─"*60}')
for i, s in enumerate(stats):
    if i == 0:
        print(f'  {0:>6}  {"(reference)":>14}  {1.0:>8.4f}  {OVERLAP:>10}')
    else:
        print(f'  {i:>6}  {s["residual_m"]:>14.4f}  {s["s"]:>8.4f}  {OVERLAP:>10}')
print(f'{"─"*60}')

residuals = [s.get('residual_m', 0.0) for s in stats[1:]]
if residuals:
    print(f'  Mean residual: {np.mean(residuals):.4f} m')
    print(f'  Max  residual: {np.max(residuals):.4f} m')

if len(residuals) > 1:
    fig_res, ax = plt.subplots(figsize=(8, 3))
    ax.bar(range(1, len(residuals)+1), residuals, color='steelblue')
    ax.set_xlabel('Chunk index')
    ax.set_ylabel('Alignment residual (m)')
    ax.set_title('Per-chunk alignment residual')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'alignment_residuals.png'), dpi=150)
    plt.show()

## 9. Chunk-Size Experiment

How does reconstruction quality and memory usage change with chunk size?

In [ ]:
CHUNK_SIZES  = [5, 8, 10, 15, 20]

cs_records = experiment_chunk_sizes(
    pipe, all_paths,
    chunk_sizes=CHUNK_SIZES,
    overlap=OVERLAP,
    conf_thresh=CONF_THRESH,
    max_frames=MAX_FRAMES,
)

print('\n=== Chunk-size results ===')
print(f'{"size":>6} {"chunks":>8} {"points":>10} {"time_s":>8} {"peak_MB":>10} {"resid_m":>10}')
print('─' * 60)
for r in cs_records:
    print(f'{r["chunk_size"]:>6} {r["n_chunks"]:>8} {r["n_points"]:>10,} '
          f'{r["total_time_s"]:>8.1f} {r["peak_gpu_mb"]:>10.0f} '
          f'{r["mean_residual_m"]:>10.4f}')

In [ ]:
if len(cs_records) > 1:
    cs_vals   = [r['chunk_size']     for r in cs_records]
    mem_vals  = [r['peak_gpu_mb']    for r in cs_records]
    time_vals = [r['total_time_s']   for r in cs_records]
    res_vals  = [r['mean_residual_m']for r in cs_records]
    pts_vals  = [r['n_points']       for r in cs_records]

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    axes[0].plot(cs_vals, [m/1024 for m in mem_vals], 'b-o')
    axes[0].axhline(16, color='r', ls='--', label='16 GB')
    axes[0].set_xlabel('Chunk size (frames)')
    axes[0].set_ylabel('Peak GPU memory (GB)')
    axes[0].set_title('Memory vs. Chunk Size')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(cs_vals, time_vals, 'g-o')
    axes[1].set_xlabel('Chunk size (frames)')
    axes[1].set_ylabel('Total inference time (s)')
    axes[1].set_title('Time vs. Chunk Size')
    axes[1].grid(True, alpha=0.3)

    axes[2].plot(cs_vals, res_vals, 'r-o')
    axes[2].set_xlabel('Chunk size (frames)')
    axes[2].set_ylabel('Mean alignment residual (m)')
    axes[2].set_title('Quality vs. Chunk Size')
    axes[2].grid(True, alpha=0.3)

    plt.suptitle('Chunk-size sweep')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'chunk_size_sweep.png'), dpi=150)
    plt.show()

## 10. Overlap Experiment

More overlap frames → better alignment, but more redundant computation.

In [ ]:
OVERLAPS = [1, 2, 3, 4]

ov_records = experiment_overlaps(
    pipe, all_paths,
    chunk_size=CHUNK_SIZE,
    overlaps=OVERLAPS,
    conf_thresh=CONF_THRESH,
    max_frames=MAX_FRAMES,
)

print('\n=== Overlap results ===')
print(f'{"overlap":>8} {"chunks":>8} {"points":>10} {"time_s":>8} {"resid_m":>10}')
print('─' * 50)
for r in ov_records:
    print(f'{r["overlap"]:>8} {r["n_chunks"]:>8} {r["n_points"]:>10,} '
          f'{r["total_time_s"]:>8.1f} {r["mean_residual_m"]:>10.4f}')

In [ ]:
if len(ov_records) > 1:
    ov_vals  = [r['overlap']          for r in ov_records]
    res_ov   = [r['mean_residual_m']  for r in ov_records]
    time_ov  = [r['total_time_s']     for r in ov_records]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
    ax1.plot(ov_vals, res_ov, 'r-o')
    ax1.set_xlabel('Overlap (frames)')
    ax1.set_ylabel('Mean alignment residual (m)')
    ax1.set_title('Alignment Quality vs. Overlap')
    ax1.grid(True, alpha=0.3)

    ax2.plot(ov_vals, time_ov, 'g-o')
    ax2.set_xlabel('Overlap (frames)')
    ax2.set_ylabel('Total inference time (s)')
    ax2.set_title('Time vs. Overlap')
    ax2.grid(True, alpha=0.3)

    plt.suptitle('Overlap sweep')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'overlap_sweep.png'), dpi=150)
    plt.show()

## 11. Memory Profiling — Per-Chunk Peaks

In [ ]:
crs  = chunked_result['chunk_results']
idxs = [cr.chunk_idx for cr in crs]
mems = [cr.peak_gpu_mb for cr in crs]
tims = [cr.inference_time_s for cr in crs]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.bar(idxs, [m/1024 for m in mems], color='steelblue')
ax1.axhline(16, color='r', ls='--', label='16 GB limit')
ax1.set_xlabel('Chunk index')
ax1.set_ylabel('Peak GPU memory (GB)')
ax1.set_title('Per-chunk Peak Memory')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.bar(idxs, tims, color='coral')
ax2.set_xlabel('Chunk index')
ax2.set_ylabel('Inference time (s)')
ax2.set_title('Per-chunk Inference Time')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'per_chunk_memory.png'), dpi=150)
plt.show()

print(f'Total inference time: {sum(tims):.2f} s')
print(f'Max per-chunk memory: {max(mems):.0f} MB  ({max(mems)/1024:.2f} GB)')
print(f'Min per-chunk memory: {min(mems):.0f} MB  ({min(mems)/1024:.2f} GB)')

## 12. Comparison: Chunked vs. Full-Batch (if available)

In [ ]:
if baseline_result is not None:
    print('=== Full-batch vs. Chunked comparison ===')
    print(f'\n  Method       Points       Time (s)   Peak (GB)')
    print(f'  ─────────────────────────────────────────────')
    print(f'  Full-batch   {len(baseline_result["point_cloud"]):>10,}  '
          f'{baseline_result["inference_time_s"]:>9.2f}  '
          f'{baseline_result["peak_gpu_mb"]/1024:>9.2f}')
    print(f'  Chunked      {len(chunked_result["point_cloud"]):>10,}  '
          f'{sum(cr.inference_time_s for cr in chunked_result["chunk_results"]):>9.2f}  '
          f'{max(cr.peak_gpu_mb for cr in chunked_result["chunk_results"])/1024:>9.2f}')

    # Side-by-side visualisation
    from src.visualization import save_ply as sv
    sv(os.path.join(OUTPUT_DIR, 'baseline_points.ply'),
       baseline_result['point_cloud'], baseline_result['point_colors'])
    sv(os.path.join(OUTPUT_DIR, 'chunked_points.ply'),
       chunked_result['point_cloud'], chunked_result['point_colors'])
    print('\n  PLY files saved for side-by-side inspection in MeshLab/CloudCompare.')
else:
    print('Full-batch baseline not available (OOM or skipped).')
    print('Saving chunked result only.')
    save_ply(
        os.path.join(OUTPUT_DIR, 'chunked_points.ply'),
        chunked_result['point_cloud'],
        chunked_result['point_colors'],
    )
    print('Chunked PLY saved.')

## 13. Summary & Recommendations

Fill in the table below from your experimental results.

In [ ]:
print('=== Phase 3 Summary ===')
print(f'  Total frames   : {len(all_paths)}')
print(f'  Chunk size     : {CHUNK_SIZE}')
print(f'  Overlap        : {OVERLAP}')
print(f'  Chunks         : {len(chunked_result["chunk_results"])}')
print(f'  Merged points  : {len(chunked_result["point_cloud"]):,}')
print(f'  Total time (s) : {sum(cr.inference_time_s for cr in chunked_result["chunk_results"]):.2f}')
print(f'  Max chunk VRAM : {max(cr.peak_gpu_mb for cr in chunked_result["chunk_results"])/1024:.2f} GB')

print()
print('Recommended settings (from experiments):')
if cs_records:
    # Recommend chunk_size where residual is lowest within 16 GB
    safe = [r for r in cs_records if r['peak_gpu_mb'] < 15_000]
    if safe:
        best = min(safe, key=lambda r: r['mean_residual_m'])
        print(f'  Best chunk_size: {best["chunk_size"]}  '
              f'(residual={best["mean_residual_m"]:.4f} m, '
              f'peak={best["peak_gpu_mb"]/1024:.2f} GB)')

if ov_records:
    best_ov = min(ov_records, key=lambda r: r['mean_residual_m'])
    print(f'  Best overlap   : {best_ov["overlap"]}  '
          f'(residual={best_ov["mean_residual_m"]:.4f} m)')

print()
print('Next steps:')
print('  → Notebook 03: IMU pre-integration for improved pose estimation')
print('  → Notebook 04: Quantitative evaluation on EuRoC / KITTI')